In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = (
    SparkSession.builder
    .appName("desafio_08_parquet")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 23:55:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Parte A — CSV → Parquet

In [4]:
df_pedidos = spark.read.csv("/workspace/datasets/08_parquet/pedidos.csv", header= True, inferSchema= True)

In [5]:
df_pedidos.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [6]:
df_pedidos.write.mode("overwrite").parquet("/workspace/datasets/08_parquet/pedidos_parquet_desafio")

In [7]:
df_pedidos_parquet = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_parquet_desafio")

Qual diferença você percebe entre recuperar o schema de um CSV e recuperar o schema de um Parquet?

Resposta: No csv precisamos avisar o spark sobre o schema atraves da função inferSchema, com isso o spark consegue indentificar a tipagem das colunas, ja com o parquet isso já não é necessario, apenas é necessario passar o caminho do arquivo para a leitura do spark e com isso ele ja consegue definir o schema e os tipos das colunas. 

Parte B — Column Pruning

In [8]:
df_column_pruning = (
    df_pedidos_parquet.select("cliente_id", "valor_total")
)

In [9]:
df_column_pruning.explain(True)

== Parsed Logical Plan ==
'Project ['cliente_id, 'valor_total]
+- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Project [cliente_id#42, valor_total#46]
+- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Optimized Logical Plan ==
Project [cliente_id#42, valor_total#46]
+- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [cliente_id#42,valor_total#46] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_parquet_desafio], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,valor_total:double>



Como o ReadSchema demonstra que ocorreu Column Pruning?

Resposta: Ele demonstra informada que so foram utilizada duas colunas, ao inves de todas do dataframe, colunas essas que estão no select()

Parte C — Predicate Pushdown

In [11]:
df_predicate_pushdown = (
    df_pedidos_parquet.select("cliente_id", "valor_total")
    .filter(F.col("valor_total") > 500)
)

In [12]:
df_predicate_pushdown.explain(True)

== Parsed Logical Plan ==
'Filter ('valor_total > 500)
+- Project [cliente_id#42, valor_total#46]
   +- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Filter (valor_total#46 > cast(500 as double))
+- Project [cliente_id#42, valor_total#46]
   +- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Optimized Logical Plan ==
Project [cliente_id#42, valor_total#46]
+- Filter (isnotnull(valor_total#46) AND (valor_total#46 > 500.0))
   +- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Physical Plan ==
*(1) Filter (isnotnull(valor_total#46) AND (valor_total#46 > 500.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [cliente_id#42,valor_total#46] Batched: true, DataFilters: [isnotnull(valor_total#46), (valor_total#46 > 500.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file

Explique com suas palavras a diferença entre:

- ReadSchema
- PushedFilters

Resposta: Readschema informa as coluans utilizadas no spark durante uma função, por exemplo se fizermos df = spark.read.csv(...) no readschema havera todas as colunas do dataframe, se fizermos um filtro selecionando coluna A e B, nesse cenario no readschame existira apenas as colunas A e B, isso é chamado de columnm pruning, quando o spark realmente utiliza as colunas necessarias. Ja PusheFilters 